# 04 — Transfer Learning and Model Selection
**PentAI | Goats and Cattle | Google Colab**

Train a **separate MobileNetV2 transfer-learning classifier** for each species, then compare each against Notebook 03's baseline using **validation loss**. No test set is loaded here. The ImageNet weights may require internet on the first run. The model metadata preserves class order and preprocessing information.

## 1. Mount Google Drive

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Local Python session detected. Edit PROJECT_ROOT in the setup cell to your local dataset folder.')

## 2. Configuration and reusable data utilities
Change `RUN_OWNER` in the first cell. For local Jupyter, change `PROJECT_ROOT` and `OUTPUT_ROOT`.

In [ ]:
from pathlib import Path
import os, json, zipfile, shutil, hashlib, random
import numpy as np
import pandas as pd
from PIL import Image
import tensorflow as tf
from IPython.display import display

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
PROJECT_ROOT = Path('/content/drive/MyDrive/Capstone Project')
DATASETS_DIR = PROJECT_ROOT / 'Datasets'
SPLIT_DIR = PROJECT_ROOT / 'outputs' / 'prepared'
WORK_DIR = Path('/content/pentai_data')
# Every member must set their own name: output is private to their MyDrive by default.
RUN_OWNER = 'CHANGE_TO_YOUR_NAME'
if RUN_OWNER == 'CHANGE_TO_YOUR_NAME':
    raise ValueError('Set RUN_OWNER to your own name before running. Do not write to shared team outputs.')
OUTPUT_ROOT = Path('/content/drive/MyDrive/PentAI_Individual_Outputs') / RUN_OWNER
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ZIP_NAMES = {'cattle': ['Cow data.zip'],
             'goat': ['Goat Pox Detection.v1i.tensorflow.zip', 'Goat Pox Detection.v1i.tensorflow(1).zip']}
IMG_SIZE=(224,224)
BATCH_SIZE=32
AUTOTUNE=tf.data.AUTOTUNE
print('TensorFlow:', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Keep the original ZIP files unchanged. Extract copies only into temporary Colab storage.
def safe_extract_zip(zip_path, target):
    target.mkdir(parents=True, exist_ok=True)
    done = target / '.extraction_complete'
    if done.exists(): return target
    with zipfile.ZipFile(zip_path) as z:
        base = target.resolve()
        for member in z.infolist():
            dest = (target / member.filename).resolve()
            if not dest.is_relative_to(base):
                raise ValueError(f'Unsafe filename in archive: {member.filename}')
        z.extractall(target)
    done.touch()
    return target

def ensure_images(animal):
    target = WORK_DIR / animal
    if (target / '.extraction_complete').exists(): return target
    zips = [DATASETS_DIR / x for x in ZIP_NAMES[animal]]
    found = next((z for z in zips if z.exists()), None)
    if found is None:
        raise FileNotFoundError(f'{animal}: expected one of {zips}. Change ZIP_NAMES if needed.')
    print('Extracting:', found.name)
    return safe_extract_zip(found, target)

def make_path_index(root):
    # Robust against ZIPs that introduce an extra top-level directory.
    paths = [p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.gif'}]
    by_tail = {}
    for p in paths:
        parts = p.relative_to(root).as_posix().split('/')
        for i in range(len(parts)):
            key = '/'.join(parts[i:]).casefold()
            by_tail.setdefault(key, set()).add(str(p))
    return by_tail

def resolve_manifest_paths(df, root):
    index = make_path_index(root)
    resolved = []
    for relative in df['relative_path'].astype(str):
        key = relative.replace('\\','/').lstrip('/').casefold()
        matches = index.get(key, set())
        if len(matches) != 1:
            raise ValueError(f'Image path must resolve uniquely ({len(matches)} matches): {relative}')
        resolved.append(next(iter(matches)))
    result = df.copy()
    result['absolute_path'] = resolved
    return result

def load_splits(animal, partitions=('train','valid')):
    items=[]
    for part in partitions:
        csv_path=SPLIT_DIR / f'{animal}_{part}.csv'
        if not csv_path.exists(): raise FileNotFoundError(f'Missing Notebook 02 output: {csv_path}')
        frame=pd.read_csv(csv_path, keep_default_na=False)
        if not {'relative_path','class','split'}.issubset(frame.columns):
            raise ValueError(f'{csv_path.name} lacks a required column')
        if frame.empty or not frame['split'].eq(part).all():
            raise ValueError(f'{csv_path.name} is empty or contains the wrong partition')
        if not frame['relative_path'].is_unique: raise ValueError(f'Duplicate paths in {csv_path.name}')
        frame['class'] = frame['class'].astype(str)
        items.append(frame)
    combined=pd.concat(items,ignore_index=True)
    if combined['relative_path'].duplicated().any(): raise ValueError('Image appears in more than one split')
    if 'sha256' in combined and (combined.groupby('sha256')['split'].nunique()>1).any():
        raise ValueError('Exact hash leakage found between splits')
    if 'similarity_group' in combined and (combined.groupby('similarity_group')['split'].nunique()>1).any():
        raise ValueError('Possible similar-image leakage found between splits')
    return resolve_manifest_paths(combined,ensure_images(animal))

def check_approval():
    # The team must first approve the Notebook 01/02 reviews; never infer approval from CSV existence.
    if not TEAM_APPROVED_SPLITS:
        raise RuntimeError('Notebook 02 splits still require team approval. Set TEAM_APPROVED_SPLITS=True ONLY after sign-off.')

def make_ds(df, classes, training=False):
    labels = {label:i for i,label in enumerate(classes)}
    unknown = set(df['class']) - set(labels)
    if unknown: raise ValueError(f'Unexpected labels: {sorted(unknown)}')
    p = df['absolute_path'].astype(str).to_numpy()
    y = np.array([labels[x] for x in df['class']],dtype=np.int32)
    ds=tf.data.Dataset.from_tensor_slices((p,y))
    def read_image(path,label):
        im = tf.io.decode_image(tf.io.read_file(path),channels=3,expand_animations=False)
        im.set_shape((None,None,3))
        im=tf.image.resize(tf.cast(im,tf.float32),IMG_SIZE)
        return im,label
    if training: ds=ds.shuffle(len(df),seed=SEED,reshuffle_each_iteration=True)
    return ds.map(read_image,num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

def save_metadata(path, meta):
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(meta,indent=2),encoding='utf-8')

AUG = "tf.keras.Sequential([tf.keras.layers.RandomFlip('horizontal'),tf.keras.layers.RandomRotation(0.08),tf.keras.layers.RandomZoom(0.10)],name='train_only_augmentation')"

## 3. Approval and settings

In [ ]:
TEAM_APPROVED_SPLITS = False  # Change only after team sign-off
HEAD_EPOCHS = 8
FINE_TUNE_EPOCHS = 5
check_approval()

## 4. Train frozen backbone, then optionally fine-tune

In [ ]:
def train_transfer(animal):
    df=load_splits(animal)
    train=df[df['split']=='train']; valid=df[df['split']=='valid']
    classes=sorted(train['class'].unique().tolist())
    if len(classes)<2 or set(valid['class'])!=set(classes):
        raise ValueError(f'{animal}: class coverage must match in train and validation')
    train_ds=make_ds(train,classes,True); val_ds=make_ds(valid,classes)
    aug=tf.keras.Sequential([tf.keras.layers.RandomFlip('horizontal'),tf.keras.layers.RandomRotation(0.08),tf.keras.layers.RandomZoom(0.10)])
    try:
        base=tf.keras.applications.MobileNetV2(input_shape=(*IMG_SIZE,3),include_top=False,weights='imagenet')
    except Exception as e:
        raise RuntimeError('Could not fetch ImageNet weights. Enable Colab internet and retry.') from e
    base.trainable=False
    inp=tf.keras.Input(shape=(*IMG_SIZE,3))
    x=aug(inp)
    x=tf.keras.applications.mobilenet_v2.preprocess_input(x)
    x=base(x,training=False)  # keep BatchNorm statistics frozen
    x=tf.keras.layers.GlobalAveragePooling2D()(x)
    x=tf.keras.layers.Dropout(.25)(x)
    out=tf.keras.layers.Dense(len(classes),activation='softmax')(x)
    model=tf.keras.Model(inp,out,name=f'{animal}_mobilenetv2')
    run=OUTPUT_ROOT/'models'/animal/'transfer'; run.mkdir(parents=True,exist_ok=True)
    best=run/'best.keras'
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint(str(best),monitor='val_loss',save_best_only=True)]
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    first=model.fit(train_ds,validation_data=val_ds,epochs=HEAD_EPOCHS,callbacks=callbacks)
    # Fine-tuning: unfreeze only final convolutional layers, NOT BatchNorm layers.
    base.trainable=True
    for layer in base.layers[:-30]: layer.trainable=False
    for layer in base.layers:
        if isinstance(layer,tf.keras.layers.BatchNormalization): layer.trainable=False
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    second=model.fit(train_ds,validation_data=val_ds,epochs=FINE_TUNE_EPOCHS,callbacks=callbacks)
    pd.concat([pd.DataFrame(first.history),pd.DataFrame(second.history)],ignore_index=True).to_csv(run/'history.csv',index=False)
    winner=tf.keras.models.load_model(best)
    vl,acc=winner.evaluate(val_ds,verbose=0)
    save_metadata(run/'metadata.json',{'animal':animal,'model_type':'transfer','architecture':'MobileNetV2',
        'classes':classes,'image_size':list(IMG_SIZE),'preprocessing':'mobilenet_v2.preprocess_input inside model',
        'validation_loss':float(vl),'validation_accuracy':float(acc),
        'split_source':'Notebook 02 approved CSVs','test_used':False,'model_file':'best.keras'})
    print(animal,'transfer validation loss:',round(vl,4))

for animal in ('cattle','goat'):
    train_transfer(animal)

## 5. Choose best candidate by validation loss
Selection is **per species**. The winning file is recorded, not copied, so both candidates remain available for audit.

In [ ]:
selection={}
for animal in ('cattle','goat'):
    choices=[]
    for variant in ('baseline','transfer'):
        folder=OUTPUT_ROOT/'models'/animal/variant
        meta_path=folder/'metadata.json'
        if not meta_path.is_file():
            print(f'Not found: {meta_path} (run Notebook 03 first for baseline)')
            continue
        metadata=json.loads(meta_path.read_text())
        choices.append((metadata['validation_loss'],variant,folder/'best.keras',metadata))
    if not choices: raise RuntimeError(f'No candidate found for {animal}')
    loss,variant,path,metadata=min(choices,key=lambda x:x[0])
    selection[animal]={'variant':variant,'model_path':str(path),
        'metadata_path':str(path.parent/'metadata.json'),'validation_loss':loss}
    print(animal,'selected:',variant,'validation loss:',round(loss,4))
save_metadata(OUTPUT_ROOT/'models'/'selection.json',selection)
print('Selection saved. Notebook 05 opens the held-out test set ONCE for this frozen selection.')

**Outputs:** separate transfer model and metadata per species, plus `models/selection.json`. Do not select or tune models using the test results.